In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from openai import OpenAI

# -----------------
# Paths
# -----------------
REP_PATH = Path("data/processed/venezuela/posts_repr.parquet")
EVAL_DIR = Path("data/evaluated/ven/hourly")
OUT_PATH = Path("data/evaluated/ven/cluster_summaries.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# -----------------
# Config
# -----------------
MODEL = "gpt-4.1-mini"  # fast + solid for summarization :contentReference[oaicite:1]{index=1}
MAX_POSTS_PER_CLUSTER = 20
MIN_POSTS_PER_CLUSTER = 4
SEED = 0

client = OpenAI(api_key="key here")


# -----------------
# Load representation layer
# -----------------
df_repr = pd.read_parquet(REP_PATH)
df_repr.columns = df_repr.columns.str.strip()
df_repr["post_id"] = df_repr["post_id"].astype(str)

# prefer "text" but fall back if your repr uses "tweet"
if "text" not in df_repr.columns and "tweet" in df_repr.columns:
    df_repr = df_repr.rename(columns={"tweet": "text"})

df_repr = df_repr[["post_id", "text"]].dropna(subset=["text"])
df_repr["text"] = df_repr["text"].astype(str)

rng = np.random.default_rng(SEED)

rows = []

# -----------------
# Iterate hourly window files
# -----------------
for fp in sorted(EVAL_DIR.glob("*.parquet")):
    df_eval = pd.read_parquet(fp)
    df_eval.columns = df_eval.columns.str.strip()

    # normalize
    df_eval["post_id"] = df_eval["post_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "is_noise" in df_eval.columns:
        df_eval = df_eval[~df_eval["is_noise"]]
    df_eval = df_eval.dropna(subset=["cluster_id"])
    if len(df_eval) == 0:
        continue

    df_eval["cluster_id"] = df_eval["cluster_id"].astype(int)
    window = str(df_eval["window"].iloc[0]) if "window" in df_eval.columns else fp.stem

    df = df_eval.merge(df_repr, on="post_id", how="left").dropna(subset=["text"])
    if len(df) == 0:
        continue

    for cid, g in df.groupby("cluster_id"):
        if len(g) < MIN_POSTS_PER_CLUSTER:
            continue

        # sample representative posts (simple random sample; swap for centroid-based if you want)
        if len(g) > MAX_POSTS_PER_CLUSTER:
            g = g.sample(MAX_POSTS_PER_CLUSTER, random_state=SEED)

        excerpts = []
        for t in g["text"].tolist():
            t = t.replace("\n", " ").strip()
            excerpts.append(t[:500])  # cap per post to control prompt size

        prompt = (
            "You are reducing a cluster of social-media posts into a single narrative summary.\n"
            "Write ONE paragraph (3–5 sentences) describing:\n"
            "- the main claim or storyline,\n"
            "- key actors/objects mentioned,\n"
            "- the implied goal or policy preference (if any),\n"
            "- notable framing (e.g., threat, sovereignty, economy),\n"
            "Avoid quoting more than a short phrase; synthesize.\n\n"
            f"Window: {window}\n"
            f"Cluster ID: {cid}\n\n"
            "Posts:\n- " + "\n- ".join(excerpts)
        )

        resp = client.responses.create(
            model=MODEL,
            input=[{
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}]
            }],
            max_output_tokens=220,
        )
        summary = resp.output_text.strip()

        rows.append({
            "window": window,
            "cluster_id": int(cid),
            "n_posts": int(len(g)),
            "summary": summary,
        })

out = pd.DataFrame(rows).sort_values(["window", "cluster_id"])
out.to_parquet(OUT_PATH, index=False)

print(f"Wrote {len(out)} cluster summaries to {OUT_PATH}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/venezuela/posts_repr.parquet'

In [1]:
!pip install -U typing_extensions
!pip install openai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
